# Preparação da base FAO-56 e do dataset MLP

Execute todas as células na ordem, em kernel novo, a partir da raiz do projeto.
A altura do anemômetro é de 10 m.
A agregação mantém a data original do INMET.
Não há imputação. A análise `clip030` usa colunas separadas e não alimenta a ETo.
Os checkpoints se referem à base de 01/01/2001 a 31/08/2026; revise a configuração ao trocar a base.
A última célula sobrescreve os dois CSVs na pasta de saída configurada.



## 1. Configuração e funções de validação

Importa pandas, NumPy e Path; define latitude, altitude, altura confirmada do anemômetro, período e pasta de saída. `exigir` interrompe o fluxo quando uma condição falha; `validar_numericas` converte valores e rejeita infinitos, preservando NaN. Os checkpoints são específicos da base atual e não substituem controle de qualidade meteorológico.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuração da estação A601 e da versão de dados auditada.
LATITUDE = -22.757778
ALTITUDE = 35
ALTURA_ANEMOMETRO_M = 10  # Confirmada pelo responsável pelo estudo.
DATA_INICIAL = "2001-01-01"
DATA_FINAL = "2026-08-31"
ROOT = Path.cwd().parent
ARQUIVO_INMET_PROCESSADO = ROOT / "dados" / "INMET_processado.csv"
ARQUIVO_INMET_RADIACAO = ROOT / "dados" / "INMET_radiacao.csv"
PASTA_SAIDA = Path("dados_FAO-56")
VALIDAR_CHECKPOINTS_REFERENCIA = True  # Rever para outra versão/período da base.

def exigir(condicao, mensagem):
    if not bool(condicao):
        raise ValueError(mensagem)

def validar_numericas(tabela, colunas):
    for coluna in colunas:
        exigir(coluna in tabela, f"Coluna obrigatória ausente: {coluna}")
        tabela[coluna] = pd.to_numeric(tabela[coluna], errors="raise")
        exigir(not np.isinf(tabela[coluna]).any(), f"Valor infinito em {coluna}")

## 2. Leitura e diagnóstico inicial da tabela diária

Lê o CSV separado por ponto e vírgula, ignorando as dez linhas iniciais de metadados. As duas consultas seguintes mostram quantidades e percentuais de ausências, sem remover registros. O cabeçalho deve continuar na posição esperada.


In [ ]:
df = pd.read_csv(ARQUIVO_INMET_PROCESSADO
    ,
    sep=";",
    skiprows=10,
    encoding="utf-8-sig"
)

In [ ]:
df.isna().sum()

In [ ]:
df.isna().mean() * 100

## 3. Nomes padronizados e inspeção das colunas

Renomeia as variáveis para nomes usados nos cálculos. `info` mostra tipos e valores presentes; os resumos de faltantes permitem identificar colunas com baixa cobertura. Esta etapa não converte unidades.


In [ ]:
df = df.rename(columns={
    "Data Medicao": "data",
    "PRECIPITACAO TOTAL, DIARIO (AUT)(mm)": "precipitacao_mm",
    "PRESSAO ATMOSFERICA MEDIA DIARIA (AUT)(mB)": "pressao_mb",
    "TEMPERATURA DO PONTO DE ORVALHO MEDIA DIARIA (AUT)(°C)": "t_orvalho_c",
    "TEMPERATURA MAXIMA, DIARIA (AUT)(°C)": "tmax_c",
    "TEMPERATURA MEDIA, DIARIA (AUT)(°C)": "tmedia_inmet_c",
    "TEMPERATURA MINIMA, DIARIA (AUT)(°C)": "tmin_c",
    "UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)": "ur_media_pct",
    "UMIDADE RELATIVA DO AR, MINIMA DIARIA (AUT)(%)": "ur_min_pct",
    "VENTO, RAJADA MAXIMA DIARIA (AUT)(m/s)": "rajada_max_ms",
    "VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)": "vento_ms"
})

In [ ]:
df.info()

df.isna().sum()

In [ ]:
faltantes = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

print(faltantes)

## 4. Colunas auxiliares e contrato numérico

Remove apenas colunas `Unnamed:` depois de confirmar que estão vazias. Preserva as variáveis meteorológicas e verifica as colunas esperadas. Não preenche ausências nem aplica limites físicos aos valores.


In [ ]:
# Só remove colunas auxiliares vazias; variáveis esperadas são preservadas.
colunas_auxiliares = [c for c in df if c.startswith("Unnamed:")]
exigir(df[colunas_auxiliares].isna().all().all(),
       "Uma coluna auxiliar contém dados: revisar antes de removê-la.")
df = df.drop(columns=colunas_auxiliares)

In [ ]:
colunas_observadas = [
    "precipitacao_mm", "pressao_mb", "t_orvalho_c", "tmax_c",
    "tmedia_inmet_c", "tmin_c", "ur_media_pct", "ur_min_pct",
    "rajada_max_ms", "vento_ms",
]
validar_numericas(df, colunas_observadas)

## 5. Calendário diário

Converte as datas, rejeita datas ausentes ou duplicadas, ordena cronologicamente e exige o calendário completo configurado. A ordenação é necessária para identificar corretamente blocos consecutivos de falhas.


In [ ]:
df["data"] = pd.to_datetime(df["data"], format="%Y-%m-%d", errors="raise")
exigir(df["data"].notna().all(), "Data diária ausente.")
exigir(not df["data"].duplicated().any(), "Datas diárias duplicadas.")
df = df.sort_values("data").reset_index(drop=True)
datas_esperadas = pd.date_range(DATA_INICIAL, DATA_FINAL, freq="D")
exigir(pd.DatetimeIndex(df["data"]).equals(datas_esperadas),
       "A base diária não corresponde ao calendário completo configurado.")

## 6. Completude meteorológica e lacunas

Seleciona temperaturas extremas, ponto de orvalho e vento para diagnosticar a disponibilidade inicial. Calcula cobertura por ano e identifica sequências de dias incompletos com início, fim e duração. A radiação ainda não participa desse diagnóstico. As flags não excluem linhas; as consultas repetidas ajudam a inspecionar a cobertura e as duplicatas.


In [ ]:
variaveis_pm = [
    "tmax_c",
    "tmin_c",
    "t_orvalho_c",
    "vento_ms"
]

In [ ]:
completos_pm = df[variaveis_pm].notna().all(axis=1)

print("Dias completos:", completos_pm.sum())
print("Dias incompletos:", (~completos_pm).sum())
print("Percentual completo:", completos_pm.mean() * 100)

In [ ]:
df["ano"] = df["data"].dt.year

faltas_ano = (
    df.groupby("ano")[variaveis_pm]
      .apply(lambda x: x.isna().sum())
)

faltas_ano

In [ ]:
df["pm_base_completa"] = df[variaveis_pm].notna().all(axis=1)

In [ ]:
df.groupby("ano")["pm_base_completa"].agg(
    dias_total="size",
    dias_completos="sum",
    proporcao="mean"
)

In [ ]:
variaveis_pm = [
    "tmax_c",
    "tmin_c",
    "t_orvalho_c",
    "vento_ms"
]

df["pm_base_completa"] = df[variaveis_pm].notna().all(axis=1)

df["falha_pm"] = ~df["pm_base_completa"]

df["grupo_falha"] = (
    df["falha_pm"] != df["falha_pm"].shift()
).cumsum()

lacunas = (
    df[df["falha_pm"]]
    .groupby("grupo_falha")
    .agg(
        inicio=("data", "min"),
        fim=("data", "max"),
        dias=("data", "size")
    )
    .sort_values("dias", ascending=False)
)

lacunas.head(20)

In [ ]:
faltas_por_ano = (
    df.groupby("ano")[variaveis_pm]
      .apply(lambda x: x.isna().sum())
)

faltas_por_ano

In [ ]:
print("Datas duplicadas:", df["data"].duplicated().sum())

## 7. Dia do ano e latitude

Cria o dia do ano (`doy`) e converte a latitude de graus para radianos com `np.deg2rad`, preparando os argumentos das funções trigonométricas.


In [ ]:
df["doy"] = df["data"].dt.dayofyear

In [ ]:
phi = np.deg2rad(LATITUDE)

## 8. Temperatura média FAO

Calcula `(Tmax + Tmin)/2` para a formulação FAO-56. Mantém `tmedia_inmet_c` separada, pois essa média observada será uma feature da MLP.


In [ ]:
df["tmean_fao_c"] = (
    df["tmax_c"] + df["tmin_c"]
) / 2

## 9. Pressões de vapor e déficit

Define a função de pressão de saturação em kPa. Calcula `es` pela média das pressões nas temperaturas extremas, `ea` pela temperatura média do ponto de orvalho e o déficit `es - ea`. As ausências propagam-se naturalmente.


In [ ]:
def esat(t):
    return 0.6108 * np.exp(
        (17.27 * t) / (t + 237.3)
    )

df["es_tmax_kpa"] = esat(df["tmax_c"])
df["es_tmin_kpa"] = esat(df["tmin_c"])

df["es_kpa"] = (
    df["es_tmax_kpa"] +
    df["es_tmin_kpa"]
) / 2

In [ ]:
df["ea_kpa"] = esat(df["t_orvalho_c"])

In [ ]:
df["vpd_kpa"] = df["es_kpa"] - df["ea_kpa"]

## 10. Pressão atmosférica, gamma e delta

Estima a pressão em kPa pela altitude da estação e calcula a constante psicrométrica em kPa/°C. A pressão observada permanece como controle, sem condicionar o alvo. Calcula a inclinação da curva de saturação na temperatura média FAO; este delta é diferente da declinação solar.


In [ ]:
P_fao = 101.3 * (
    (293 - 0.0065 * ALTITUDE) / 293
) ** 5.26

P_fao

In [ ]:
df["pressao_fao_kpa"] = P_fao

In [ ]:
df["gamma_kpa_c"] = (
    0.000665 * df["pressao_fao_kpa"]
)

In [ ]:
T = df["tmean_fao_c"]

df["delta_kpa_c"] = (
    4098 *
    (0.6108 * np.exp(
        (17.27 * T) / (T + 237.3)
    ))
    /
    ((T + 237.3) ** 2)
)

## 11. Leitura e inspeção da radiação horária

Carrega o segundo CSV do INMET e inspeciona estrutura, primeiras linhas, cabeçalhos e faltantes. A energia de cada intervalo horário está em kJ/m²; ainda não é o total diário.


In [ ]:
df_rad=pd.read_csv(ARQUIVO_INMET_RADIACAO,
                sep=";",
                    skiprows=10,
                    encoding="utf-8-sig")

In [ ]:
df_rad.info()

In [ ]:
df_rad.head(30)

In [ ]:
print(df_rad.columns.tolist())

In [ ]:
df_rad.isna().sum()

## 12. Validação da grade horária

Valida as horas 0000 a 2300, constrói timestamps sem deslocar o fuso e rejeita duplicatas. Exige as mesmas datas da tabela diária e 24 horários distintos por dia. Isso valida a estrutura: os valores de radiação ainda podem ser NaN. As consultas seguintes mostram exemplos da conversão e a contagem de linhas diárias.


In [ ]:
# Mantém a data/hora fornecida pelo INMET, sem conversão de fuso.
# A janela do produto diário permanece uma definição documental do provedor.
horas = pd.to_numeric(df_rad["Hora Medicao"], errors="raise")
exigir(horas.notna().all() and horas.isin(range(0, 2400, 100)).all(),
       "Hora ausente ou fora da grade 0000, 0100, ..., 2300.")
df_rad["hora"] = horas.astype(int).astype(str).str.zfill(4)
df_rad["datetime"] = pd.to_datetime(
    df_rad["Data Medicao"].astype(str) + " " + df_rad["hora"],
    format="%Y-%m-%d %H%M", errors="raise",
)
exigir(df_rad["datetime"].notna().all(), "Timestamp horário ausente.")
exigir(not df_rad["datetime"].duplicated().any(), "Timestamp horário duplicado.")
df_rad = df_rad.sort_values("datetime").reset_index(drop=True)
df_rad["data"] = df_rad["datetime"].dt.normalize()
exigir(pd.DatetimeIndex(df_rad["data"].drop_duplicates()).equals(datas_esperadas),
       "As datas da radiação não correspondem ao calendário diário.")
exigir(df_rad.groupby("data")["datetime"].nunique().eq(24).all(),
       "Cada dia deve conter 24 horas distintas, mesmo quando a radiação é NaN.")

In [ ]:
df_rad[
    ["Data Medicao", "Hora Medicao", "hora", "datetime"]
].head(30)

In [ ]:
print(df_rad[["Data Medicao", "Hora Medicao"]].head(10))
print()
print(df_rad[["Data Medicao", "Hora Medicao"]].dtypes)
print()
print(df_rad["Hora Medicao"].head(10).tolist())

In [ ]:
df_rad[
    ["Data Medicao", "Hora Medicao", "hora", "datetime"]
].head(24)

In [ ]:
horas_por_dia = (
    df_rad.groupby("Data Medicao")
    .size()
)

print(horas_por_dia.value_counts().sort_index())

## 13. Padronização e diagnóstico da radiação bruta

Renomeia a coluna de radiação, verifica valores numéricos e resume extremos, ausências, zeros e negativos. Mostra também um dia de exemplo. Nenhum desses diagnósticos corrige ou remove observações.


In [ ]:
df_rad = df_rad.rename(columns={
    "RADIACAO GLOBAL(Kj/m²)": "radiacao_kj_m2"
})
validar_numericas(df_rad, ["radiacao_kj_m2"])

In [ ]:
print(df_rad["radiacao_kj_m2"].describe())

print(
    "NaN:",
    df_rad["radiacao_kj_m2"].isna().sum()
)

print(
    "Zeros:",
    (df_rad["radiacao_kj_m2"] == 0).sum()
)

print(
    "Negativos:",
    (df_rad["radiacao_kj_m2"] < 0).sum())

In [ ]:
dia_teste = df_rad[
    df_rad["Data Medicao"] == "2020-01-15"
]

dia_teste[
    ["Hora Medicao", "datetime", "radiacao_kj_m2"]
]

## 14. Controle de qualidade e distribuição de falhas

Cria uma coluna QC com negativos limitados a zero, mantendo intacta a coluna bruta e preservando NaN. Os blocos seguintes contam valores válidos por dia, faltantes e negativos por hora, distribuição das faltas diárias e magnitude dos negativos. A interpretação como offset instrumental é uma hipótese de processamento, não uma calibração comprovada do sensor.


In [ ]:
df_rad["radiacao_qc_kj_m2"] = (
    df_rad["radiacao_kj_m2"]
    .clip(lower=0)
)

In [ ]:
print("NaN:", df_rad["radiacao_kj_m2"].isna().sum())
print("Negativos:", (df_rad["radiacao_kj_m2"] < 0).sum())

In [ ]:
validos_por_dia = (
    df_rad.groupby("Data Medicao")["radiacao_kj_m2"]
    .count()
)

print(validos_por_dia.value_counts().sort_index())

In [ ]:
resumo_hora = (
    df_rad.groupby("Hora Medicao")["radiacao_kj_m2"]
    .agg(
        total="size",
        faltantes=lambda x: x.isna().sum(),
        negativos=lambda x: (x < 0).sum(),
        media="mean",
        mediana="median"
    )
)

resumo_hora["pct_faltantes"] = (
    resumo_hora["faltantes"] /
    resumo_hora["total"] * 100
)

resumo_hora

In [ ]:
faltantes_por_dia = (
    df_rad.groupby("Data Medicao")["radiacao_kj_m2"]
    .apply(lambda x: x.isna().sum())
)

print(
    faltantes_por_dia
    .value_counts()
    .sort_index()
)

In [ ]:
negativos = df_rad.loc[
    df_rad["radiacao_kj_m2"] < 0,
    "radiacao_kj_m2"
]

print(negativos.describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
))

## 15. Agregação diária e conversão de energia

Soma a radiação QC por data. `min_count=1` impede que um grupo totalmente ausente vire zero. Só aceita o total se houver 24 valores presentes e divide por 1.000 para converter kJ em MJ. A soma estabelece a escala diária: não se divide por 24 nem se multiplica por 3.600. `rad_completa=False` indica dia incompleto; Rs fica NaN. Em seguida, resume Rs e cobertura anual.


In [ ]:
# Soma QC sem substituir grupos inteiramente ausentes por zero.
# As 24 horas distintas já foram verificadas; faltas meteorológicas são mantidas.
rad_diaria = (
    df_rad.groupby("data")
    .agg(
        n_rad=("radiacao_kj_m2", "count"),
        rs_kj_m2=("radiacao_qc_kj_m2", lambda s: s.sum(min_count=1)),
    ).reset_index()
)
rad_diaria["rad_completa"] = rad_diaria["n_rad"].eq(24)
rad_diaria["rs_mj_m2_dia"] = (
    rad_diaria["rs_kj_m2"].div(1000).where(rad_diaria["rad_completa"])
)
exigir(df_rad["radiacao_kj_m2"].isna().equals(df_rad["radiacao_qc_kj_m2"].isna()),
       "O controle de qualidade alterou a máscara de valores ausentes.")

In [ ]:
rad_diaria["rs_mj_m2_dia"].describe()

In [ ]:
rad_diaria["ano"] = rad_diaria["data"].dt.year

resumo_rad_ano = (
    rad_diaria.groupby("ano")
    .agg(
        dias_total=("data", "size"),
        dias_rad_validos=("rad_completa", "sum")
    )
)

resumo_rad_ano["proporcao"] = (
    resumo_rad_ano["dias_rad_validos"]
    / resumo_rad_ano["dias_total"]
)

resumo_rad_ano

## 16. Integração das duas tabelas

Faz merge à esquerda, validado como um-para-um, e verifica preservação das datas e cobertura horária. As inspeções seguintes confirmam que Rs existe exatamente quando a radiação está completa. A flag `eto_base_completa` passa a incluir radiação, mas não elimina dias.


In [ ]:
datas_antes_merge = df["data"].copy()
df = df.merge(
    rad_diaria[["data", "rs_mj_m2_dia", "n_rad", "rad_completa"]],
    on="data", how="left", validate="one_to_one", sort=False,
)
exigir(df["data"].equals(datas_antes_merge), "O merge alterou as datas ou linhas.")
exigir(df[["n_rad", "rad_completa"]].notna().all().all(),
       "Há datas diárias sem correspondência na tabela horária.")
exigir(df["rs_mj_m2_dia"].notna().equals(df["rad_completa"]),
       "Rs deve existir exatamente nos dias de radiação completa.")

In [ ]:
df.info()

In [ ]:
print(df["rad_completa"].value_counts())

print(
    "Rs válido quando rad_completa=True:",
    df.loc[df["rad_completa"], "rs_mj_m2_dia"].notna().sum()
)

print(
    "Rs indevido quando rad_completa=False:",
    df.loc[~df["rad_completa"], "rs_mj_m2_dia"].notna().sum()
)

In [ ]:
variaveis_para_eto = [
    "tmax_c",
    "tmin_c",
    "t_orvalho_c",
    "vento_ms",
    "rs_mj_m2_dia"
]

df["eto_base_completa"] = (
    df[variaveis_para_eto]
    .notna()
    .all(axis=1)
)

print(df["eto_base_completa"].value_counts())

print(
    "Percentual utilizável:",
    df["eto_base_completa"].mean() * 100
)

## 17. Geometria solar e radiação extraterrestre

Usa dia do ano e latitude para calcular distância relativa Terra–Sol, declinação solar, ângulo do pôr do sol e Ra. Emprega Gsc = 0,0820 MJ m⁻² min⁻¹ e o fator de minutos por dia; Ra resulta em MJ m⁻² dia⁻¹.


In [ ]:
J = df["doy"]

df["dr"] = (
    1 + 0.033 * np.cos(2 * np.pi * J / 365)
)

In [ ]:
df["delta_solar"] = (
    0.409 *
    np.sin(
        (2 * np.pi * J / 365) - 1.39
    )
)

In [ ]:
df["ws"] = np.arccos(
    -np.tan(phi) *
    np.tan(df["delta_solar"])
)

In [ ]:
GSC = 0.0820  # MJ m-2 min-1

df["ra_mj_m2_dia"] = (
    (24 * 60 / np.pi)
    * GSC
    * df["dr"]
    * (
        df["ws"]
        * np.sin(phi)
        * np.sin(df["delta_solar"])
        +
        np.cos(phi)
        * np.cos(df["delta_solar"])
        * np.sin(df["ws"])
    )
)

## 18. Céu claro, ondas curtas e razão de radiação

Estima Rso com altitude e Ra; calcula Rns = 0,77 Rs, com albedo 0,23. Preserva a razão observada Rs/Rso para diagnóstico e cria a razão principal limitada apenas acima a 1. As estatísticas seguintes verificam sua distribuição e valores fora do intervalo esperado.


In [ ]:
df["rso_mj_m2_dia"] = (
    (0.75 + 2e-5 * ALTITUDE)
    * df["ra_mj_m2_dia"]
)

In [ ]:
df["rns_mj_m2_dia"] = (
    0.77 * df["rs_mj_m2_dia"]
)

In [ ]:
df["rs_rso"] = (
    df["rs_mj_m2_dia"] /
    df["rso_mj_m2_dia"]
)
# Formulação principal: somente limite superior, sem clip inferior.
df["rs_rso_fao"] = df["rs_rso"].clip(upper=1.0)

In [ ]:
df["rs_rso"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
)

In [ ]:
print(
    "Rs/Rso > 1:",
    (df["rs_rso"] > 1).sum()
)

print(
    "Rs/Rso <= 0:",
    (df["rs_rso"] <= 0).sum()
)

## 19. Ondas longas e saldo de radiação principal

Converte Tmax e Tmin para Kelvin somando 273,16, convenção numérica da FAO-56, antes da quarta potência. Usa sigma = 4,903 × 10⁻⁹ MJ K⁻⁴ m⁻² dia⁻¹ e calcula Rnl; depois, Rn = Rns − Rnl. Os diagnósticos mostram distribuições, negativos e disponibilidade. Os Rnl negativos da formulação são mantidos, não substituídos automaticamente.


In [ ]:
SIGMA = 4.903e-9  # MJ K^-4 m^-2 dia^-1

tmax_k = df["tmax_c"] + 273.16
tmin_k = df["tmin_c"] + 273.16

df["rnl_mj_m2_dia"] = (
    SIGMA
    * ((tmax_k**4 + tmin_k**4) / 2)
    * (0.34 - 0.14 * np.sqrt(df["ea_kpa"]))
    * (1.35 * df["rs_rso_fao"] - 0.35)
)

In [ ]:
df["rn_mj_m2_dia"] = (
    df["rns_mj_m2_dia"]
    - df["rnl_mj_m2_dia"]
)

In [ ]:
df[
    [
        "rs_mj_m2_dia",
        "rso_mj_m2_dia",
        "rns_mj_m2_dia",
        "rnl_mj_m2_dia",
        "rn_mj_m2_dia"
    ]
].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
)

In [ ]:
print(
    "Rn < 0:",
    (df["rn_mj_m2_dia"] < 0).sum()
)

print(
    "Rnl < 0:",
    (df["rnl_mj_m2_dia"] < 0).sum()
)

print(
    "Rn válidos:",
    df["rn_mj_m2_dia"].notna().sum()
)

## 20. Análise de sensibilidade clip030

Cria uma alternativa que limita Rs/Rso entre 0,30 e 1,00. Calcula Rnl e Rn em colunas exclusivas `clip030`, sem substituir a versão principal nem alimentar o alvo. Compara os diagnósticos da alternativa e da formulação principal; plausibilidade numérica não demonstra qual versão é fisicamente mais precisa.


In [ ]:
# Sensibilidade independente: estas colunas nunca alimentam o alvo.
df["flag_rs_rso_baixo"] = df["rs_rso"] < 0.30
df["rs_rso_clip030"] = df["rs_rso"].clip(lower=0.30, upper=1.0)

In [ ]:
SIGMA = 4.903e-9

tmax_k = df["tmax_c"] + 273.16
tmin_k = df["tmin_c"] + 273.16

df["rnl_clip030_mj_m2_dia"] = (
    SIGMA
    * ((tmax_k**4 + tmin_k**4) / 2)
    * (0.34 - 0.14 * np.sqrt(df["ea_kpa"]))
    * (1.35 * df["rs_rso_clip030"] - 0.35)
)

df["rn_clip030_mj_m2_dia"] = (
    df["rns_mj_m2_dia"]
    - df["rnl_clip030_mj_m2_dia"]
)

In [ ]:
print("Rs/Rso < 0.30:", df["flag_rs_rso_baixo"].sum())
print("Rnl < 0:", (df["rnl_clip030_mj_m2_dia"] < 0).sum())
print("Rn < 0:", (df["rn_clip030_mj_m2_dia"] < 0).sum())
print("Rn válidos:", df["rn_clip030_mj_m2_dia"].notna().sum())

df[
    ["rnl_clip030_mj_m2_dia", "rn_clip030_mj_m2_dia"]
].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
)

In [ ]:
print("Rnl < 0:", (df["rnl_mj_m2_dia"] < 0).sum())
print("Rn < 0:", (df["rn_mj_m2_dia"] < 0).sum())
print("Rn válidos:", df["rn_mj_m2_dia"].notna().sum())

## 21. Correção da altura do vento

Converte o vento medido a 10 m em vento equivalente a 2 m, usando logaritmo natural. O fator é aproximadamente 0,747951. Trata-se de correção de altura; a unidade continua m/s.


In [ ]:
df["u2_ms"] = (
    df["vento_ms"]
    * 4.87
    / np.log(67.8 * ALTURA_ANEMOMETRO_M - 5.42)
)

## 22. ETo diária pelo método FAO-56

Combina os termos radiativo e aerodinâmico da equação diária, com G = 0, temperatura média FAO, Rn principal, vento a 2 m e déficit de vapor. O resultado é `eto_pmt`, em mm/dia. Ausências nas entradas necessárias produzem ausência no alvo.


In [ ]:
# Escala diária: G = 0; T é a média FAO, não a média INMET.
df["eto_pmt"] = (
    (
        0.408
        * df["delta_kpa_c"]
        * df["rn_mj_m2_dia"]
    )
    +
    (
        df["gamma_kpa_c"]
        * (900 / (df["tmean_fao_c"] + 273))
        * df["u2_ms"]
        * (df["es_kpa"] - df["ea_kpa"])
    )
) / (
    df["delta_kpa_c"]
    +
    df["gamma_kpa_c"]
    * (1 + 0.34 * df["u2_ms"])
)

## 23. Diagnóstico final da ETo

Conta ETo válidas, negativas e nulas; mostra extremos e quantis. Investiga quais entradas faltam nos dias com Rn válido mas ETo ausente e resume a cobertura anual. Esses resultados descrevem a saída; não são uma validação contra evapotranspiração medida.


In [ ]:
print(
    "ETo válidas:",
    df["eto_pmt"].notna().sum()
)

print(
    "ETo negativas:",
    (df["eto_pmt"] < 0).sum()
)

print(
    "ETo = 0:",
    (df["eto_pmt"] == 0).sum()
)

In [ ]:
print(
    "ETo máxima:",
    df["eto_pmt"].max()
)

print(
    "ETo mínima:",
    df["eto_pmt"].min()
)

In [ ]:
df["eto_pmt"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

In [ ]:
variaveis_eto = [
    "rn_mj_m2_dia",
    "delta_kpa_c",
    "gamma_kpa_c",
    "tmean_fao_c",
    "u2_ms",
    "es_kpa",
    "ea_kpa"
]

df.loc[
    df["rn_mj_m2_dia"].notna() &
    df["eto_pmt"].isna(),
    variaveis_eto
].isna().sum()

In [ ]:
resumo_eto_ano = (
    df.groupby("ano")
    .agg(
        dias_total=("data", "size"),
        dias_eto=("eto_pmt", lambda x: x.notna().sum())
    )
)

resumo_eto_ano["proporcao"] = (
    resumo_eto_ano["dias_eto"] /
    resumo_eto_ano["dias_total"]
)

resumo_eto_ano

## 24. Dataset MLP e checkpoints de referência

Seleciona data, seis features e alvo; somente agora remove linhas incompletas. Todos os cenários devem usar essas mesmas linhas. Verifica valores finitos, propagação de ausências e checkpoints da versão atual: 9.374 dias, 8.538 Rs, 8.374 Rn, 8.331 ETo e 8.329 linhas MLP. Os checkpoints detectam alterações no processamento, mas não certificam os sensores. Existem 31 registros MLP com média INMET inferior à mínima, sinalizados na revisão de qualidade e ainda não excluídos.


In [ ]:
features = [
    "rn_mj_m2_dia",
    "u2_ms",
    "ur_media_pct",
    "tmax_c",
    "tmin_c",
    "tmedia_inmet_c"
]

target = "eto_pmt"

df_mlp = (
    df[
        ["data"] + features + [target]
    ]
    .dropna()
    .copy()
)

print(df_mlp.shape)
# Todos os cenários devem usar estas mesmas linhas, sem novo filtro por cenário.
exigir(np.isfinite(df_mlp[features + [target]].to_numpy()).all(),
       "O dataset MLP contém valores não finitos.")
exigir(df["eto_pmt"].notna().equals(df[variaveis_eto].notna().all(axis=1)),
       "A completude da ETo diverge da completude de suas entradas.")

if VALIDAR_CHECKPOINTS_REFERENCIA:
    observados = {
        "dias": len(df), "horas": len(df_rad),
        "rad_ausente": int(df_rad["radiacao_kj_m2"].isna().sum()),
        "rad_negativa": int(df_rad["radiacao_kj_m2"].lt(0).sum()),
        "rs_validos": int(df["rs_mj_m2_dia"].notna().sum()),
        "rn_validos": int(df["rn_mj_m2_dia"].notna().sum()),
        "eto_validas": int(df["eto_pmt"].notna().sum()),
        "rnl_negativos": int(df["rnl_mj_m2_dia"].lt(0).sum()),
        "rn_negativos": int(df["rn_mj_m2_dia"].lt(0).sum()),
        "razao_baixa": int(df["flag_rs_rso_baixo"].sum()),
        "razao_acima_1": int(df["rs_rso"].gt(1).sum()),
        "razao_nao_positiva": int(df["rs_rso"].le(0).sum()),
        "eto_nao_positiva": int(df["eto_pmt"].le(0).sum()),
        "mlp": len(df_mlp),
    }
    esperados = dict(dias=9374, horas=224976, rad_ausente=10916,
                     rad_negativa=83021, rs_validos=8538, rn_validos=8374,
                     eto_validas=8331, rnl_negativos=820, rn_negativos=0,
                     razao_baixa=1058, razao_acima_1=0, razao_nao_positiva=0,
                     eto_nao_positiva=0, mlp=8329)
    for nome, esperado in esperados.items():
        exigir(observados[nome] == esperado,
               f"Checkpoint {nome}: esperado {esperado}, obtido {observados[nome]}.")
    perdas = df["rn_mj_m2_dia"].notna() & df["eto_pmt"].isna()
    exigir(perdas.sum() == 43 and df.loc[perdas, "u2_ms"].isna().all()
           and df.loc[perdas, [v for v in variaveis_eto if v != "u2_ms"]].notna().all().all(),
           "As 43 perdas adicionais de ETo devem decorrer somente de vento ausente.")
    exigir(np.allclose([df["eto_pmt"].min(), df["eto_pmt"].max()],
                       [0.6414942257872781, 8.342549603880846], rtol=0, atol=1e-9),
           "Os extremos da ETo divergem da referência.")
    print("Todos os checkpoints da base de referência conferem.")

## 25. Exportação dos produtos

Grava a base diária completa e o dataset MLP na pasta configurada, sem índice, com separador vírgula. A base completa conserva dias incompletos, intermediários e flags; a série horária bruta continua no arquivo-fonte. Os nomes fixos sobrescrevem exportações anteriores. Esta etapa não treina a MLP nem altera a configuração de `evapo_final.ipynb`.


In [ ]:
# A exportação só ocorre depois das validações acima.
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
df.to_csv(PASTA_SAIDA / "base_processamento_fao56_A601.csv", index=False)
df_mlp.to_csv(PASTA_SAIDA / "dataset_A601_mlp.csv", index=False)

### Referência metodológica

Allen, R. G.; Pereira, L. S.; Raes, D.; Smith, M. (1998). *Crop evapotranspiration: Guidelines for computing crop water requirements*. FAO Irrigation and Drainage Paper 56.